# Notebook 04 — Chromosomes, Genes, and Genome Scale

**Module:** 05 — Biology Fundamentals  
**Notebook:** 04 of 18  
**Prerequisites:** NB03  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

When you align sequencing reads to a reference genome, you need to know *what* you're
aligning to and *why* only ~1.5% of the human genome actually codes for proteins.
When a variant caller reports 'chr7:117,548,628 G>A', you need to know how to interpret
that coordinate — what chromosome, what gene, what consequence.

---
## Step 3 — Biological Background

**Chromosomes:**
- Humans have 46 chromosomes: 22 pairs of autosomes + 2 sex chromosomes (XX or XY)
- Diploid (2n = 46): two copies of each autosome
- Gametes (sperm/egg) are haploid (n = 23)
- A **locus** is a position on a chromosome. An **allele** is one version of the sequence at that locus.

**Gene structure (eukaryotic):**
```
5'UTR  Exon1  Intron1  Exon2  Intron2  Exon3  3'UTR
─────────────────────────────────────────────────────── chromosome
       ←──── mRNA after splicing ────────────────►
```
- **Exons:** sequences retained in mature mRNA (coding sequence + UTRs)
- **Introns:** intervening sequences spliced out
- **Promoter:** ~1–2 kb upstream, where RNA polymerase binds
- **Enhancers:** can be far upstream/downstream or in introns — regulate transcription

**The human genome by the numbers:**

| Feature | Value |
|---------|-------|
| Total size | ~3.2 billion bp (haploid) |
| Protein-coding genes | ~20,000 |
| Total transcribed (RNA) | ~70–80% |
| Protein-coding sequence | ~1.5% |
| Repetitive elements | ~50% |
| Average gene size | ~27 kb |
| Average exon size | ~170 bp |
| Average intron size | ~3,400 bp |

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Cell 6.1 — Genome statistics across species
species_data = {
    'E. coli': {'genome_mb': 4.6, 'genes': 4300, 'ploidy': 1, 'gc': 50.8},
    'Yeast (S. cerevisiae)': {'genome_mb': 12, 'genes': 6600, 'ploidy': 2, 'gc': 38.3},
    'Fruit fly (D. melanogaster)': {'genome_mb': 180, 'genes': 14000, 'ploidy': 2, 'gc': 42.9},
    'Zebrafish': {'genome_mb': 1400, 'genes': 26000, 'ploidy': 2, 'gc': 36.6},
    'Mouse': {'genome_mb': 2700, 'genes': 22000, 'ploidy': 2, 'gc': 41.8},
    'Human': {'genome_mb': 3200, 'genes': 20000, 'ploidy': 2, 'gc': 41.0},
    'Wheat (T. aestivum)': {'genome_mb': 17000, 'genes': 107000, 'ploidy': 6, 'gc': 45.3},
}

print(f"{'Species':<30} {'Genome (Mb)':>12} {'Genes':>8} {'Ploidy':>7} {'GC%':>6}")
print("-" * 65)
for sp, d in species_data.items():
    print(f"  {sp:<28} {d['genome_mb']:>12,.0f} {d['genes']:>8,} {d['ploidy']:>7} {d['gc']:>5.1f}")
print()
print("Note: genome size does NOT correlate with complexity (C-value paradox)")

In [ ]:
# Cell 6.2 — Human genome composition
genome_composition = {
    'Protein-coding\nexons': 1.5,
    'Non-coding RNA': 5.0,
    'Regulatory\n(enhancers, promoters)': 8.0,
    'Introns': 26.5,
    'Repetitive elements\n(SINE, LINE, LTR)': 50.0,
    'Other / unclassified': 9.0,
}

print("Human genome composition:")
for category, pct in genome_composition.items():
    bar = '█' * int(pct / 2)
    print(f"  {category.replace(chr(10),' '):<40} {pct:>5.1f}%  {bar}")

In [ ]:
# Cell 6.3 — GC content of a DNA sequence
def gc_content(seq: str) -> float:
    """Compute GC content of a DNA/RNA sequence."""
    seq = seq.upper()
    gc = seq.count('G') + seq.count('C')
    return gc / len(seq) if len(seq) > 0 else 0.0

def sliding_window_gc(seq: str, window: int = 100) -> np.ndarray:
    """GC content in sliding windows across a sequence."""
    return np.array([gc_content(seq[i:i+window]) for i in range(len(seq) - window + 1)])

# Simulate a synthetic 'chromosome' with a CpG island (high GC region)
rng = np.random.default_rng(42)
def random_dna(n, gc=0.42):
    bases = rng.choice(['A', 'T', 'G', 'C'], size=n,
                       p=[(1-gc)/2, (1-gc)/2, gc/2, gc/2])
    return ''.join(bases)

seq = random_dna(5000, gc=0.42)  # background GC = 42%
# CpG island: positions 2000-2500 have high GC (65%)
cpg_island = random_dna(500, gc=0.65)
seq = seq[:2000] + cpg_island + seq[2500:]

gc_profile = sliding_window_gc(seq, window=100)
print(f"Overall GC content: {gc_content(seq):.2%}")
print(f"CpG island GC:      {gc_content(seq[2000:2500]):.2%}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Genome size vs. gene count + GC profile
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Genome size vs. gene count (C-value paradox)
ax = axes[0]
genomes_mb = [d['genome_mb'] for d in species_data.values()]
gene_counts = [d['genes'] for d in species_data.values()]
sp_labels = list(species_data.keys())
scatter = ax.scatter(genomes_mb, gene_counts, s=80, color='steelblue', alpha=0.8)
for i, (x, y, label) in enumerate(zip(genomes_mb, gene_counts, sp_labels)):
    ax.annotate(label.split('(')[0].strip(), (x, y),
                textcoords='offset points', xytext=(5, 3), fontsize=7)
ax.set_xscale('log')
ax.set_xlabel('Genome size (Mb, log scale)', fontsize=10)
ax.set_ylabel('Number of protein-coding genes', fontsize=10)
ax.set_title('C-value paradox:\nlarger genome ≠ more genes')

# Panel 2: GC content profile showing CpG island
ax = axes[1]
positions = np.arange(len(gc_profile))
ax.plot(positions, gc_profile, color='steelblue', lw=0.8, alpha=0.8)
ax.axhline(0.42, color='gray', ls='--', lw=1, label='Background GC (42%)')
ax.axvspan(2000, 2500, alpha=0.2, color='tomato', label='CpG island (65% GC)')
ax.set_xlabel('Position (bp)'); ax.set_ylabel('GC content (100 bp window)')
ax.set_title('Sliding-window GC content: CpG island visible')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `gc_content(seq)` and `sliding_window_gc(seq, window)` from scratch.
   Apply them to the CFTR gene (NM_000492): compute overall GC and identify any
   high-GC windows (potential CpG islands).
2. What is the difference between a gene and a protein? How many human genes produce
   proteins? What do the other genes do?
3. What is ploidy? A triploid organism has 3 copies of each chromosome.
   What fraction of loci would be heterozygous if one chromosome carries an alternate allele?
4. Why does the human genome contain ~50% repetitive elements? What are LINE and
   SINE elements? What do they have to do with genome size across species?

---
## Quiz — Active Recall

1. How many chromosomes does a human cell have? How many in a gamete?
2. What is the difference between an exon and an intron?
3. What fraction of the human genome codes for protein? What does the rest do?
4. What is the C-value paradox? Give one example.
5. What is a CpG island? Why is its GC content unusual, and what is its relevance
   to gene regulation?

---
## Reflection

**Date completed:** ____________________

> *[If you open a genome browser (UCSC or Ensembl) and navigate to chr7:117,548,628,
> what do you expect to see? What would you look for to determine whether this position
> is in an exon, intron, or intergenic region?]*

---
*Next: `05_mendelian_genetics_punnett_squares.ipynb`*